# 02. Подготовка Parquet для RRUFF и opXRD

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("неверный путь.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from project_paths import *
ensure_project_directories()

print("Project root:", PROJECT_ROOT)

Project root: D:\Users\user\Desktop\DS_XRD_project


## RRUFF: парсинг метаданных и спектров

In [2]:
import json
import re
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# PATHS

JSON_PATH = RAW_SOURCES["rruff_json"]

RAW_DIR = str(RAW_SPECTRA_DIR)

PARQUET_PATH = SOURCE_DIR / "rruff.parquet"

# RRUFF powder patterns were measured with the Cu Kalpha doublet.
CU_K_ALPHA_1 = 1.54056
CU_K_ALPHA_2 = 1.54439


os.makedirs(RAW_DIR, exist_ok=True)

In [4]:
# LOAD JSON

print("Загрузка RRUFF...")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Загружено образцов: {len(data)}")

Загрузка RRUFF...
Загружено образцов: 1362


In [5]:
# CELL PARAMETERS PARSER

def parse_cell_parameters(cell_string):
    """
    Извлекает:

    a
    b
    c
    alpha
    beta
    gamma
    volume
    crystal_system

    из строки вида:

    a: 4.2244(9) b: 6.9297(7) c: 7.8653(6)
    alpha: 90 beta: 99.65(1) gamma: 90
    volume: 226.99(4) crystal system: monoclinic
    """

    if not cell_string:
        return {
            "a": None,
            "b": None,
            "c": None,
            "alpha": None,
            "beta": None,
            "gamma": None,
            "volume": None,
            "crystal_system": None,
        }

    result = {
        "a": None,
        "b": None,
        "c": None,
        "alpha": None,
        "beta": None,
        "gamma": None,
        "volume": None,
        "crystal_system": None,
    }


    number_pattern = r"([-+]?\d+(?:\.\d+)?)(?:\(\d+\))?"

    fields = [
        "a",
        "b",
        "c",
        "alpha",
        "beta",
        "gamma",
        "volume",
    ]

    for field in fields:

        pattern = rf"{field}\s*:\s*{number_pattern}"

        match = re.search(
            pattern,
            cell_string,
            flags=re.IGNORECASE
        )

        if match:
            result[field] = float(match.group(1))

    # --------------------------------------------------------
    # Crystal system
    # --------------------------------------------------------

    match = re.search(
        r"crystal\s+system\s*:\s*([A-Za-z]+)",
        cell_string,
        flags=re.IGNORECASE
    )

    if match:
        result["crystal_system"] = match.group(1).lower()

    return result

In [6]:
# ELEMENT NORMALIZATION

VALID_ELEMENTS = {
    'H', 'He',
    'Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne',
    'Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'Ar',
    'K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe',
    'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As', 'Se',
    'Br', 'Kr', 'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo',
    'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn',
    'Sb', 'Te', 'I', 'Xe', 'Cs', 'Ba', 'La', 'Ce',
    'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy',
    'Ho', 'Er', 'Tm', 'Yb', 'Lu', 'Hf', 'Ta', 'W',
    'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb',
    'Bi', 'Po', 'At', 'Rn', 'Fr', 'Ra', 'Ac', 'Th',
    'Pa', 'U', 'Np', 'Pu', 'Am', 'Cm', 'Bk', 'Cf',
    'Es', 'Fm', 'Md', 'No', 'Lr'
}


def normalize_elements(sample):
    """
    Возвращает корректный список химических элементов.

    Для Bavenite и Tourmaline исходные:

        formula = "na"
        elements = ["n", "a"]

    считаются отсутствующими.
    """

    formula = sample.get("formula")

    if formula is None:
        return None

    if str(formula).strip().lower() == "na":
        return None

    elements = sample.get("elements")

    if not elements:
        return None

    valid = [
        element
        for element in elements
        if element in VALID_ELEMENTS
    ]

    if not valid:
        return None

    return valid

In [7]:
# BUILD DATASET

rows = []

print("\nОбработка спектров...")


for idx, sample in enumerate(data):

    rruff_id = sample["##RRUFFID"]

    # --------------------------------------------------------
    # Spectrum
    # --------------------------------------------------------

    x = np.asarray(sample["x"], dtype=np.float32)
    y = np.asarray(sample["y"], dtype=np.float32)

    if len(x) != len(y):
        raise ValueError(
            f"{rruff_id}: x/y имеют разную длину"
        )

    # --------------------------------------------------------
    # Cell parameters
    # --------------------------------------------------------

    cell = parse_cell_parameters(
        sample.get("##CELL PARAMETERS")
    )

    a = cell["a"]
    b = cell["b"]
    c = cell["c"]

    alpha = cell["alpha"]
    beta = cell["beta"]
    gamma = cell["gamma"]

    crystal_system = cell["crystal_system"]

    has_lattice = all(
        value is not None
        for value in [
            a, b, c,
            alpha, beta, gamma
        ]
    )

    # --------------------------------------------------------
    # Chemistry
    # --------------------------------------------------------

    formula = sample.get("formula")

    if formula is not None:
        formula = str(formula).strip()

    if formula and formula.lower() == "na":
        formula = None

    elements_list = normalize_elements(sample)

    has_elements = (
        elements_list is not None
        and len(elements_list) > 0
    )

    # --------------------------------------------------------
    # Save raw spectrum
    #
    # 2 x N:
    # row 0 -> x
    # row 1 -> y
    # --------------------------------------------------------

    spectrum_filename = f"rruff_{rruff_id}.npy"

    spectrum_path_abs = os.path.join(
        RAW_DIR,
        spectrum_filename
    )

    spectrum = np.vstack([x, y]).astype(np.float32)

    np.save(
        spectrum_path_abs,
        spectrum
    )

    # --------------------------------------------------------
    # Save plot
    # --------------------------------------------------------

    plot_filename = f"rruff_{rruff_id}.png"

    plot_dir = FIGURES_DIR / "02_prepare_real_parquets" / "rruff_spectra"
    plot_dir.mkdir(parents=True, exist_ok=True)
    plot_path_abs = str(plot_dir / plot_filename)

    plt.figure(figsize=(10, 5))

    plt.plot(x, y)

    plt.xlabel("2θ (degrees)")
    plt.ylabel("Intensity")

    plt.title(
        f"{sample['##NAMES']} ({rruff_id})"
    )

    plt.tight_layout()

    plt.savefig(
        plot_path_abs,
        dpi=150
    )

    plt.close()

    # --------------------------------------------------------
    # Relative path for Parquet
    # --------------------------------------------------------

    raw_spectrum_path = os.path.join(
        "raw",
        spectrum_filename
    )

    # Convert Windows separators to /
    raw_spectrum_path = raw_spectrum_path.replace(
        "\\",
        "/"
    )

    # --------------------------------------------------------
    # Phase information
    #
    # RRUFF sample = one mineral / one phase
    # --------------------------------------------------------

    phase_count = 1

    phase_fraction = 1.0

    phase_compositions = (
        [formula]
        if formula is not None
        else None
    )

    is_single_phase = True

    # --------------------------------------------------------
    # Lattice array
    # --------------------------------------------------------

    lattices_all = (
        [[
            a,
            b,
            c,
            alpha,
            beta,
            gamma
        ]]
        if has_lattice
        else None
    )

    # --------------------------------------------------------
    # Build row
    # --------------------------------------------------------

    row = {
        "sample_id": idx,

        "source": f"rruff_{rruff_id}",

        "dataset_role": "RRUFF",

        "raw_spectrum_path": raw_spectrum_path,

        "primary_wavelength": CU_K_ALPHA_1,

        "secondary_wavelength": CU_K_ALPHA_2,

        "lattice_a": a,

        "lattice_b": b,

        "lattice_c": c,

        "alpha": alpha,

        "beta": beta,

        "gamma": gamma,

        "spacegroup_number": None,

        "crystal_system": crystal_system,

        "elements": elements_list,

        "phase_count": phase_count,

        "phase_fraction": phase_fraction,

        "has_lattice": has_lattice,

        "has_spacegroup": False,

        "has_elements": has_elements,

        "has_phase_count": True,

        "has_phase_fraction": True,

        "phase_compositions": phase_compositions,

        "spacegroups_all": None,

        "lattices_all": lattices_all,

        "is_single_phase": is_single_phase,

        "is_simulated": False,

        "crystallite_size_nm": None,

        "temp_K": None,

        "elements_list": elements_list,

        "elements_json": (
            json.dumps(
                elements_list,
                ensure_ascii=False
            )
            if elements_list is not None
            else None
        )
    }

    rows.append(row)

    if (idx + 1) % 100 == 0:
        print(
            f"Обработано: {idx + 1}/{len(data)}"
        )


Обработка спектров...
Обработано: 100/1362
Обработано: 200/1362
Обработано: 300/1362
Обработано: 400/1362
Обработано: 500/1362
Обработано: 600/1362
Обработано: 700/1362
Обработано: 800/1362
Обработано: 900/1362
Обработано: 1000/1362
Обработано: 1100/1362
Обработано: 1200/1362
Обработано: 1300/1362


In [8]:
# DATAFRAME
df_rruff = pd.DataFrame(rows)

In [9]:
# COLUMN ORDER

columns = [
    "sample_id",
    "source",
    "dataset_role",
    "raw_spectrum_path",
    "primary_wavelength",
    "secondary_wavelength",
    "lattice_a",
    "lattice_b",
    "lattice_c",
    "alpha",
    "beta",
    "gamma",
    "spacegroup_number",
    "crystal_system",
    "elements",
    "phase_count",
    "phase_fraction",
    "has_lattice",
    "has_spacegroup",
    "has_elements",
    "has_phase_count",
    "has_phase_fraction",
    "phase_compositions",
    "spacegroups_all",
    "lattices_all",
    "is_single_phase",
    "is_simulated",
    "crystallite_size_nm",
    "temp_K",
    "elements_list",
    "elements_json"
]

df_rruff = df_rruff[columns]

In [10]:
# SAVE PARQUET

df_rruff.to_parquet(
    PARQUET_PATH,
    index=False
)

In [11]:
# VALIDATION

print("\n" + "=" * 60)
print("RRUFF PARQUET СОЗДАН")
print("=" * 60)

print(f"\nФайл:")
print(PARQUET_PATH)

print(f"\nСтрок: {len(df_rruff)}")
print(f"Колонок: {len(df_rruff.columns)}")

print("\nРазмеры:")
print(df_rruff.shape)

print("\nКолонки:")
print(df_rruff.columns.tolist())

print("\n=== LABEL AVAILABILITY ===")

for col in [
    "lattice_a",
    "lattice_b",
    "lattice_c",
    "alpha",
    "beta",
    "gamma",
    "crystal_system",
    "elements",
    "phase_compositions"
]:

    print(
        f"{col:<25} "
        f"{df_rruff[col].notna().sum():>5} / "
        f"{len(df_rruff)}"
    )


print("\n=== CRYSTAL SYSTEM ===")

print(
    df_rruff["crystal_system"]
    .value_counts(dropna=False)
)


print("\n=== ELEMENTS ===")

print(
    f"С elements: "
    f"{df_rruff['has_elements'].sum()}"
)


print("\n=== LATTICE ===")

print(
    f"С lattice: "
    f"{df_rruff['has_lattice'].sum()}"
)


print("\n=== SINGLE PHASE ===")

print(
    df_rruff["is_single_phase"].value_counts(
        dropna=False
    )
)


print("\n=== ПЕРВЫЕ 5 СТРОК ===")

print(
    df_rruff.head().to_string()
)


print("\n=== PARQUET READ TEST ===")

test_df = pd.read_parquet(
    PARQUET_PATH
)

print(
    f"Успешно прочитан обратно: "
    f"{test_df.shape}"
)


print("\nГотово.")


RRUFF PARQUET СОЗДАН

Файл:
D:\Users\user\Desktop\DS_XRD_project\data\source\rruff.parquet

Строк: 1362
Колонок: 31

Размеры:
(1362, 31)

Колонки:
['sample_id', 'source', 'dataset_role', 'raw_spectrum_path', 'primary_wavelength', 'secondary_wavelength', 'lattice_a', 'lattice_b', 'lattice_c', 'alpha', 'beta', 'gamma', 'spacegroup_number', 'crystal_system', 'elements', 'phase_count', 'phase_fraction', 'has_lattice', 'has_spacegroup', 'has_elements', 'has_phase_count', 'has_phase_fraction', 'phase_compositions', 'spacegroups_all', 'lattices_all', 'is_single_phase', 'is_simulated', 'crystallite_size_nm', 'temp_K', 'elements_list', 'elements_json']

=== LABEL AVAILABILITY ===
lattice_a                  1298 / 1362
lattice_b                  1298 / 1362
lattice_c                  1298 / 1362
alpha                      1298 / 1362
beta                       1298 / 1362
gamma                      1298 / 1362
crystal_system             1298 / 1362
elements                   1360 / 1362
phase_c

In [12]:
df_rruff.to_parquet(SOURCE_DIR / "df_rruff_summary_final_clean.parquet", index=False)
print("Saved RRUFF source parquet")

Saved RRUFF source parquet


## opXRD: извлечение спектров и исходных полей

### Данные и промежуточные вычисления

In [13]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import hashlib


OPXRD_ROOT = RAW_SOURCES["opxrd_root"]
OUTPUT_DIR = INTERIM_DIR
RAW_DIR = RAW_SPECTRA_DIR
OUTPUT_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

PARQUET_FILE = SOURCE_DIR / "summary.parquet"

### Функция `extract_lattice`

In [ ]:
def extract_lattice(lattice_str):
    """Парсим строку вида '(a, b, c, alpha, beta, gamma)' в список чисел."""
    if not lattice_str:
        return None
    cleaned = lattice_str.strip('()').replace(' ', '').split(',')
    if len(cleaned) != 6:
        return None
    try:
        return [float(x) for x in cleaned]
    except ValueError:
        return None

### Функция `normalize_composition`

In [ ]:
def normalize_composition(comp_str):
    
    """Оставляем строку состава как есть."""
    return comp_str if comp_str else None

### Функция `get_phase_info`

In [ ]:
def get_phase_info(phase_json):

    """Извлекаем из фазы: lattice (список из 6 чисел), spacegroup, composition, phase_fraction."""
    phase = json.loads(phase_json) if isinstance(phase_json, str) else phase_json
    lat = extract_lattice(phase.get('lattice'))
    sg = phase.get('spacegroup')
    comp = phase.get('chemical_composition')
    frac = phase.get('phase_fraction')
    return lat, sg, comp, frac

### Функция `parse_opxrd`

In [ ]:
def parse_opxrd():

    files = list(OPXRD_ROOT.rglob("pattern_*.json"))
    print(f"Found {len(files)} files.")

    records = []
    for idx, filepath in enumerate(tqdm(files, desc="Processing")):
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                item = json.load(f)
        except Exception as e:
            print(f"Error reading {filepath}: {e}")
            continue


        theta = item.get('two_theta_values', [])
        intensities = item.get('intensities', [])
        if not theta or not intensities:
            continue


        sample_id = f"opxrd_{idx+1:05d}"
        raw_path = RAW_DIR / f"{sample_id}.npy"

        spectrum = np.column_stack((theta, intensities))
        np.save(raw_path, spectrum)


        label_raw = item.get('label')
        if not label_raw:

            rec = {
                'sample_id': sample_id,
                'source': 'opXRD',
                'dataset_role': None,
                'raw_spectrum_path': str(raw_path.relative_to(OUTPUT_DIR)),
                'processed_spectrum_id': sample_id,
                'primary_wavelength': None,
                'secondary_wavelength': None,
                'lattice_a': None,
                'lattice_b': None,
                'lattice_c': None,
                'alpha': None,
                'beta': None,
                'gamma': None,
                'spacegroup_number': None,
                'spacegroup_symbol': None,
                'crystal_system': None,
                'elements': None,
                'phase_count': 0,
                'phase_fraction': None,
                'has_lattice': False,
                'has_spacegroup': False,
                'has_elements': False,
                'has_phase_count': True,
                'has_phase_fraction': False,
                'phase_compositions': None,
                'space_groups_all': None,
                'lattices_all': None,
                'other_metadata': None
            }
            records.append(rec)
            continue

        try:
            label = json.loads(label_raw)

        except:
            continue

        xray = {}
        try:
            xray = json.loads(label.get('xray_info', '{}'))

        except:
            pass

        primary_wl = xray.get('primary_wavelength')
        secondary_wl = xray.get('secondary_wavelength')

        phases_raw = label.get('phases', [])
        phase_count = len(phases_raw)

        all_lattices = []
        all_spacegroups = []
        all_compositions = []
        all_fractions = []

        has_lattice = False
        has_spacegroup = False
        has_composition = False
        has_phase_fraction = False

        for pjson in phases_raw:
            lat, sg, comp, frac = get_phase_info(pjson)

            if lat:
                all_lattices.append(lat)
                has_lattice = True

            if sg:
                all_spacegroups.append(sg)
                has_spacegroup = True

            if comp:
                all_compositions.append(comp)
                has_composition = True
                
            if frac is not None:
                all_fractions.append(frac)
                has_phase_fraction = True

        first_lat = all_lattices[0] if all_lattices else None
        first_sg = all_spacegroups[0] if all_spacegroups else None
        first_comp = all_compositions[0] if all_compositions else None


        rec = {
            'sample_id': sample_id,
            'source': 'opXRD',
            'dataset_role': None,
            'raw_spectrum_path': str(raw_path.relative_to(OUTPUT_DIR)),
            'processed_spectrum_id': sample_id,
            'primary_wavelength': primary_wl,
            'secondary_wavelength': secondary_wl,
            'lattice_a': first_lat[0] if first_lat else None,
            'lattice_b': first_lat[1] if first_lat else None,
            'lattice_c': first_lat[2] if first_lat else None,
            'alpha': first_lat[3] if first_lat else None,
            'beta': first_lat[4] if first_lat else None,
            'gamma': first_lat[5] if first_lat else None,
            'spacegroup_number': first_sg,
            'spacegroup_symbol': None,  
            'crystal_system': None,    
            'elements': first_comp,    
            'phase_count': phase_count,
            'phase_fraction': json.dumps(all_fractions) if all_fractions else None,
            'has_lattice': has_lattice,
            'has_spacegroup': has_spacegroup,
            'has_elements': has_composition,
            'has_phase_count': True,
            'has_phase_fraction': has_phase_fraction,
            'phase_compositions': json.dumps(all_compositions) if all_compositions else None,
            'space_groups_all': json.dumps(all_spacegroups) if all_spacegroups else None,
            'lattices_all': json.dumps(all_lattices) if all_lattices else None,
            'other_metadata': json.dumps({
                'is_simulated': label.get('is_simulated'),
                'crystallite_size_nm': label.get('crystallite_size_nm'),
                'temp_K': label.get('temp_K'),
                'xray_info': xray,
                'basis': None,
            }) if label else None
        }

        records.append(rec)

    df = pd.DataFrame(records)
    df.to_parquet(PARQUET_FILE, index=False)
    print(f"Saved {len(df)} records to {PARQUET_FILE}")
    return df

### Запуск и сохранение результатов

In [18]:
if __name__ == "__main__":
    df = parse_opxrd()
    print(df.info())
    print(df.head())

Found 92552 files.


Processing: 100%|██████████| 92552/92552 [06:32<00:00, 236.07it/s] 


Saved 92552 records to D:\Users\user\Desktop\DS_XRD_project\data\source\summary.parquet
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92552 entries, 0 to 92551
Data columns (total 28 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   sample_id              92552 non-null  object 
 1   source                 92552 non-null  object 
 2   dataset_role           0 non-null      object 
 3   raw_spectrum_path      92552 non-null  object 
 4   processed_spectrum_id  92552 non-null  object 
 5   primary_wavelength     91722 non-null  object 
 6   secondary_wavelength   22658 non-null  object 
 7   lattice_a              1377 non-null   float64
 8   lattice_b              1377 non-null   float64
 9   lattice_c              1377 non-null   float64
 10  alpha                  1377 non-null   float64
 11  beta                   1377 non-null   float64
 12  gamma                  1377 non-null   float64
 13  spacegroup_number 

In [19]:
df = parse_opxrd()

Found 92552 files.


Processing: 100%|██████████| 92552/92552 [09:39<00:00, 159.58it/s] 


Saved 92552 records to D:\Users\user\Desktop\DS_XRD_project\data\source\summary.parquet


## opXRD: производные признаки и crystal system

In [20]:
import pandas as pd
import numpy as np
import json
from pathlib import Path


df = pd.read_parquet(SOURCE_DIR / "summary.parquet")


df['is_single_phase'] = df['phase_count'] == 1


df.drop(columns=['processed_spectrum_id', 'spacegroup_symbol'], inplace=True, errors='ignore')

def extract_other(meta_json):
    if pd.isna(meta_json) or meta_json is None:
        return None, None, None
    try:
        meta = json.loads(meta_json)
        is_sim = meta.get('is_simulated')
        crystallite = meta.get('crystallite_size_nm')
        temp = meta.get('temp_K')
        return is_sim, crystallite, temp
    except:
        return None, None, None

df[['is_simulated', 'crystallite_size_nm', 'temp_K']] = df['other_metadata'].apply(
    lambda x: pd.Series(extract_other(x))
)
df.drop(columns=['other_metadata'], inplace=True)


df['primary_wavelength'] = pd.to_numeric(df['primary_wavelength'], errors='coerce')
df['secondary_wavelength'] = pd.to_numeric(df['secondary_wavelength'], errors='coerce')


def determine_crystal_system(row):
    a, b, c = row['lattice_a'], row['lattice_b'], row['lattice_c']
    alpha, beta, gamma = row['alpha'], row['beta'], row['gamma']
    if pd.isna(a) or pd.isna(b) or pd.isna(c) or pd.isna(alpha) or pd.isna(beta) or pd.isna(gamma):
        return None
    tol = 1e-2
    # Кубическая
    if (abs(a-b) <= tol and abs(b-c) <= tol and
        abs(alpha-90) <= tol and abs(beta-90) <= tol and abs(gamma-90) <= tol):
        return 'cubic'
    # Тетрагональная
    if (abs(a-b) <= tol and abs(alpha-90) <= tol and abs(beta-90) <= tol and abs(gamma-90) <= tol):
        return 'tetragonal'
    # Гексагональная
    if (abs(a-b) <= tol and abs(alpha-90) <= tol and abs(beta-90) <= tol and abs(gamma-120) <= tol):
        return 'hexagonal'
    # Орторомбическая
    if (abs(alpha-90) <= tol and abs(beta-90) <= tol and abs(gamma-90) <= tol):
        return 'orthorhombic'
    # Моноклинная
    if (abs(alpha-90) <= tol and abs(gamma-90) <= tol and abs(beta-90) > tol):
        return 'monoclinic'

    return 'triclinic'

df['crystal_system'] = df.apply(determine_crystal_system, axis=1)

df['elements_all'] = df['phase_compositions'] 

df.rename(columns={'space_groups_all': 'spacegroups_all'}, inplace=True)

output_path = SOURCE_DIR / "summary_enhanced.parquet"
df.to_parquet(output_path, index=False)
print(f"Enhanced dataset saved to {output_path}")
print(df.info())
print("\nSample rows:")
print(df.head())

Enhanced dataset saved to D:\Users\user\Desktop\DS_XRD_project\data\source\summary_enhanced.parquet
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92552 entries, 0 to 92551
Data columns (total 30 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   sample_id             92552 non-null  object 
 1   source                92552 non-null  object 
 2   dataset_role          0 non-null      object 
 3   raw_spectrum_path     92552 non-null  object 
 4   primary_wavelength    91722 non-null  float64
 5   secondary_wavelength  22658 non-null  float64
 6   lattice_a             1377 non-null   float64
 7   lattice_b             1377 non-null   float64
 8   lattice_c             1377 non-null   float64
 9   alpha                 1377 non-null   float64
 10  beta                  1377 non-null   float64
 11  gamma                 1377 non-null   float64
 12  spacegroup_number     611 non-null    object 
 13  crystal_system       

## opXRD: извлечение химических элементов

In [ ]:
import pandas as pd
import json
import re

ELEMENTS_SET = {
    'H', 'He', 'Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne',
    'Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'Ar', 'K', 'Ca',
    'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn',
    'Ga', 'Ge', 'As', 'Se', 'Br', 'Kr', 'Rb', 'Sr', 'Y', 'Zr',
    'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn',
    'Sb', 'Te', 'I', 'Xe', 'Cs', 'Ba', 'La', 'Ce', 'Pr', 'Nd',
    'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb',
    'Lu', 'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg',
    'Tl', 'Pb', 'Bi', 'Po', 'At', 'Rn', 'Fr', 'Ra', 'Ac', 'Th',
    'Pa', 'U', 'Np', 'Pu', 'Am', 'Cm', 'Bk', 'Cf', 'Es', 'Fm',
    'Md', 'No', 'Lr', 'Rf', 'Db', 'Sg', 'Bh', 'Hs', 'Mt', 'Ds',
    'Rg', 'Cn', 'Nh', 'Fl', 'Mc', 'Lv', 'Ts', 'Og'
}

def extract_elements_from_formula(formula):
    """
    Извлекаем уникальные валидные химические элементы из строки формулы.
    Ищет заглавную букву, за которой следует 0 или 1 строчная.
    """
    if not isinstance(formula, str) or not formula:
        return []
    tokens = re.findall(r'[A-Z][a-z]?', formula)
    elements = [tok for tok in tokens if tok in ELEMENTS_SET]
    seen = set()
    unique = []
    for el in elements:
        if el not in seen:
            seen.add(el)
            unique.append(el)
    return unique

def extract_elements_from_phase_compositions(json_str):
    """
    Принимаемм JSON-строку со списком составов и возвращает список уникальных элементов.
    """
    if pd.isna(json_str) or json_str is None:
        return None
    try:
        comps = json.loads(json_str)
        if not isinstance(comps, list):
            return None
        all_elements = []
        for comp in comps:
            all_elements.extend(extract_elements_from_formula(comp))
        seen = set()
        unique = []
        for el in all_elements:
            if el not in seen:
                seen.add(el)
                unique.append(el)
        return unique
    except:
        return extract_elements_from_formula(json_str)

df = pd.read_parquet(SOURCE_DIR / "summary_enhanced.parquet")

def get_elements(row):
    if pd.notna(row['phase_compositions']) and row['phase_compositions'] is not None:
        return extract_elements_from_phase_compositions(row['phase_compositions'])
    elif pd.notna(row['elements']) and row['elements'] is not None:
        return extract_elements_from_formula(row['elements'])
    else:
        return None

df['elements_list'] = df.apply(get_elements, axis=1)
df['elements_json'] = df['elements_list'].apply(lambda x: json.dumps(x) if x is not None else None)

print(df[df['elements_list'].notna()][['sample_id', 'elements', 'phase_compositions', 'elements_list']].head(10))

all_elements = set()
for lst in df['elements_list'].dropna():
    all_elements.update(lst)
print("Все уникальные элементы:", sorted(all_elements))

        sample_id                 elements  \
1052  opxrd_01053                     PbI2   
1053  opxrd_01054                     PbI2   
1054  opxrd_01055                     PbI2   
1055  opxrd_01056  CH5N2PbI3 (alpha phase)   
1056  opxrd_01057  CH5N2PbI3 (alpha phase)   
1057  opxrd_01058  CH5N2PbI3 (alpha phase)   
1058  opxrd_01059  CH5N2PbI3 (alpha phase)   
1059  opxrd_01060  CH5N2PbI3 (alpha phase)   
1060  opxrd_01061  CH5N2PbI3 (alpha phase)   
1061  opxrd_01062  CH5N2PbI3 (alpha phase)   

                       phase_compositions     elements_list  
1052                    ["PbI2", "PbBr2"]       [Pb, I, Br]  
1053                    ["PbI2", "PbBr2"]       [Pb, I, Br]  
1054                    ["PbI2", "PbBr2"]       [Pb, I, Br]  
1055  ["CH5N2PbI3 (alpha phase)", "PbI2"]  [C, H, N, Pb, I]  
1056  ["CH5N2PbI3 (alpha phase)", "PbI2"]  [C, H, N, Pb, I]  
1057  ["CH5N2PbI3 (alpha phase)", "PbI2"]  [C, H, N, Pb, I]  
1058  ["CH5N2PbI3 (alpha phase)", "PbI2"]  [C, H, N, Pb, I]

In [22]:
df.to_parquet(SOURCE_DIR / "summary_final_with_elements.parquet", index=False)

## opXRD: финальная очистка списков

In [23]:
import pandas as pd
import json
import numpy as np

df = pd.read_parquet(SOURCE_DIR / "summary_final_with_elements.parquet")

if 'elements_all' in df.columns:
    df.drop(columns=['elements_all'], inplace=True)
    print("Колонка elements_all удалена.")

def fix_elements_json(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    if isinstance(val, str):
        try:
            lst = json.loads(val)
        except:
            return val 
    elif isinstance(val, list):
        lst = val
    else:
        return None
    filtered = [x for x in lst if x is not None and x != 'nan']
    if not filtered:
        return None
    return json.dumps(filtered)

df['elements_json'] = df['elements_json'].apply(fix_elements_json)

def fix_elements_list(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    if isinstance(val, list):
        filtered = [x for x in val if x is not None and x != 'nan']
        return filtered if filtered else None
    if isinstance(val, str):
        try:
            lst = json.loads(val)
            filtered = [x for x in lst if x is not None and x != 'nan']
            return filtered if filtered else None
        except:

            return None
    if isinstance(val, np.ndarray):
        val_list = val.tolist()
        filtered = [x for x in val_list if x is not None and x != 'nan']
        return filtered if filtered else None
    return None

df['elements_list'] = df['elements_list'].apply(fix_elements_list)


print("Проверка elements_json:")
print(df['elements_json'].value_counts(dropna=False).head(10))

print("\nПроверка elements_list:")
len_counts = df['elements_list'].apply(lambda x: len(x) if isinstance(x, list) else None).value_counts(dropna=False)
print(len_counts.head(10))

df.to_parquet(SOURCE_DIR / "df_opxrd_summary_final_clean.parquet", index=False)
print("Таблица сохранена как summary_final_clean.parquet")

Колонка elements_all удалена.
Проверка elements_json:
elements_json
None                                91615
["Zn", "V"]                           315
["Pb", "Br", "Cl"]                    163
["Al"]                                135
["C", "H", "N", "Pb", "I"]             81
["Pb", "I", "Br"]                      66
["Zn"]                                 45
["V"]                                  45
["Cs", "C", "H", "N", "Pb", "I"]       41
["Cu", "Al"]                           34
Name: count, dtype: int64

Проверка elements_list:
elements_list
NaN    91615
2.0      352
3.0      230
1.0      225
5.0       85
6.0       45
Name: count, dtype: int64
Таблица сохранена как summary_final_clean.parquet


In [24]:
import pandas as pd
import json

df = pd.read_parquet(SOURCE_DIR / "df_opxrd_summary_final_clean.parquet")

def fix_phase_compositions(val):
    if pd.isna(val) or val is None:
        return None
    try:
        lst = json.loads(val)
        filtered = [x for x in lst if x != 'nan' and x is not None]
        if not filtered:
            return None
        return json.dumps(filtered)
    except:
        return val

df['phase_compositions'] = df['phase_compositions'].apply(fix_phase_compositions)


print("Проверка phase_compositions после очистки:")
print(df['phase_compositions'].value_counts(dropna=False).head(10))

df.to_parquet(SOURCE_DIR / "df_opxrd_summary_final_clean.parquet", index=False)
print("Таблица окончательно очищена.")

Проверка phase_compositions после очистки:
phase_compositions
None                                    91615
["PbBr2", "PbCl2"]                        163
["Al"]                                    135
["PbI2", "PbBr2"]                          66
["VNx"]                                    45
["ZnNx"]                                   45
["CH3NHPbI3", "PbI2"]                      41
["Csx(CH5N2)1-xPbI3and alpha phase"]       41
["CH5N2PbI3 (alpha phase)", "PbI2"]        40
["Cu", "Al"]                               34
Name: count, dtype: int64
Таблица окончательно очищена.


## Проверка полного совпадения формата

In [25]:
expected_columns = [
    "sample_id", "source", "dataset_role", "raw_spectrum_path",
    "primary_wavelength", "secondary_wavelength",
    "lattice_a", "lattice_b", "lattice_c", "alpha", "beta", "gamma",
    "spacegroup_number", "crystal_system", "elements", "phase_count",
    "phase_fraction", "has_lattice", "has_spacegroup", "has_elements",
    "has_phase_count", "has_phase_fraction", "phase_compositions",
    "spacegroups_all", "lattices_all", "is_single_phase", "is_simulated",
    "crystallite_size_nm", "temp_K", "elements_list", "elements_json",
]
rruff = pd.read_parquet(SOURCE_DIR / "df_rruff_summary_final_clean.parquet")
opxrd = pd.read_parquet(SOURCE_DIR / "df_opxrd_summary_final_clean.parquet")
assert list(rruff.columns) == expected_columns
assert list(opxrd.columns) == expected_columns
assert len(rruff) == 1362, len(rruff)
assert len(opxrd) == 92552, len(opxrd)
print("FORMAT_OK: both datasets have 31 columns and expected row counts")

FORMAT_OK: both datasets have 31 columns and expected row counts


## Вывод

Две исходные таблицы и 93 914 real `.npy`-спектров созданы внутри проекта.